# M2 — DeepAR on Colab (the first rung in the heavy environment)

M0 (seasonal-naive) and M1 (ARIMA) run on a laptop. **M2 DeepAR** is the first model that needs the *heavy group* (GluonTS + PyTorch Lightning), so we run it on Colab.

This notebook is a thin driver — all the logic lives in the repo (`src/models/deepar.py`, `experiments/run_deepar.py`). It clones the repo, installs the heavy group, lets the data auto-download, trains DeepAR on the **Exchange** train split, scores the **test** windows with the *same* metrics module as M0/M1, and appends one row to `results/registry.csv`.

**Before you start:** `Runtime → Change runtime type → GPU` (Exchange is tiny, so CPU also works in a couple of minutes; the runner uses `--accelerator auto`, which picks the GPU when present).

## 1. Clone the sandbox repo

In [1]:
!git clone -q https://github.com/Icaica14/pml-diffusion-tsf-test.git
%cd pml-diffusion-tsf-test

/content/pml-diffusion-tsf-test


## 2. Install the heavy group

`gluonts[torch]` pulls a compatible PyTorch + Lightning. We pin `numpy<2.0` to match the data contract (`requirements.txt`). This takes a minute or two.

In [ ]:
!pip install -q "gluonts[torch]" "numpy<2.0" pandas pyyaml requests

In [ ]:
# Sanity check: versions + whether a GPU is visible.
import gluonts, torch, numpy as np
print('gluonts', gluonts.__version__, '| torch', torch.__version__, '| numpy', np.__version__)
print('CUDA available:', torch.cuda.is_available())

## 3. Train + evaluate M2

The Exchange series auto-downloads on first use (the runner's `download()` is idempotent). Training is one global DeepAR over the 8 channels; the test windows are then forecast with no refit (each window conditioned only on its own context — the same leakage-free protocol as M1).

`--epochs 30` is a reasonable budget for Exchange; bump it if the loss is still falling.

In [ ]:
!python -m experiments.run_deepar --epochs 30 --samples 100 --accelerator auto

## 4. Inspect the result and bring the row home

The cloned repo's `results/registry.csv` already holds the committed M0 and M1 rows; the cell above appended **M2**. Read it back to compare the three rungs side by side, then download the file and replace your local `results/registry.csv` (or copy just the new `deepar` row into it) and commit it from your laptop.

In [ ]:
import pandas as pd
df = pd.read_csv('results/registry.csv')
cols = ['model', 'MASE', 'CRPS', 'cov50', 'cov90', 'width90', 'fit_s', 'predict_s']
print(df[[c for c in cols if c in df.columns]].to_string(index=False))

In [ ]:
# Download the updated registry so you can commit it from your laptop.
from google.colab import files
files.download('results/registry.csv')